In [2]:
import pandas as pd
import json

# We load only 100,000 reviews — enough to train a great model
# Professional trick: never load 6GB when 100k rows does the job

data = []
with open("../data/yelp_academic_dataset_review.json", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 100000:
            break
        data.append(json.loads(line))

df = pd.DataFrame(data)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (100000, 9)
Columns: ['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny', 'cool', 'text', 'date']


In [3]:
# See the first 5 rows
df.head()

,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5.0,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3.0,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5.0,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03
4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4.0,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15


In [4]:
# Check for missing values
df.isnull().sum()

review_id      0
user_id        0
business_id    0
stars          0
useful         0
funny          0
cool           0
text           0
date           0
dtype: int64

In [5]:
# How are star ratings distributed?
df['stars'].value_counts().sort_index()

stars
1.0    10921
2.0     7988
3.0    11362
4.0    25337
5.0    44392
Name: count, dtype: int64

In [6]:
# Reviews with 0 useful votes AND extreme ratings = potentially fake
# Reviews with useful votes > 0 = likely genuine

df['label'] = df['useful'].apply(lambda x: 0 if x > 0 else 1)
# 0 = genuine, 1 = suspicious/fake

print(df['label'].value_counts())

label
1    59858
0    40142
Name: count, dtype: int64


In [7]:
# See an example of each
print("--- GENUINE REVIEW ---")
print(df[df['label'] == 0]['text'].iloc[0])
print("\n--- SUSPICIOUS REVIEW ---")
print(df[df['label'] == 1]['text'].iloc[0])

--- GENUINE REVIEW ---
I've taken a lot of spin classes over the years, and nothing compares to the classes at Body Cycle. From the nice, clean space and amazing bikes, to the welcoming and motivating instructors, every class is a top notch work out.

For anyone who struggles to fit workouts in, the online scheduling system makes it easy to plan ahead (and there's no need to line up way in advanced like many gyms make you do).

There is no way I can write this review without giving Russell, the owner of Body Cycle, a shout out. Russell's passion for fitness and cycling is so evident, as is his desire for all of his clients to succeed. He is always dropping in to classes to check in/provide encouragement, and is open to ideas and recommendations from anyone. Russell always wears a smile on his face, even when he's kicking your butt in class!

--- SUSPICIOUS REVIEW ---
If you decide to eat here, just be aware it is going to take about 2 hours from beginning to end. We have tried it multi

In [10]:
# Save only the columns we need for the project
df_clean = df[['text', 'stars', 'label']].copy()

# Save it as a CSV so we never have to reload the 6GB file again
df_clean.to_csv("../data/reviews_clean.csv", index=False)

print("Saved!", df_clean.shape)
print(df_clean.head(3))

Saved! (100000, 3)
                                                text  stars  label
0  If you decide to eat here, just be aware it is...    3.0      1
1  I've taken a lot of spin classes over the year...    5.0      0
2  Family diner. Had the buffet. Eclectic assortm...    3.0      1


In [15]:
import pandas as pd
import json

# Reload original data
data = []
with open("../data/yelp_academic_dataset_review.json", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 100000:
            break
        data.append(json.loads(line))

df = pd.DataFrame(data)

# Label using relaxed rules to get more fake samples
def better_label(row):
    word_count = len(str(row['text']).split())
    is_extreme_rating = row['stars'] in [1.0, 5.0]
    is_useless = row['useful'] == 0
    is_short = word_count < 40

    if is_extreme_rating and is_useless and is_short:
        return 1  # fake
    elif row['useful'] >= 1:
        return 0  # genuine
    else:
        return 0  # default genuine

df['label'] = df.apply(better_label, axis=1)

print("Before balancing:")
print(df['label'].value_counts())

# ── UNDERSAMPLING ──
# Take all fake reviews
fake_df = df[df['label'] == 1]

# Take same number of genuine reviews randomly
genuine_df = df[df['label'] == 0].sample(n=len(fake_df), random_state=42)

# Combine and shuffle
balanced_df = pd.concat([fake_df, genuine_df]).sample(frac=1, random_state=42)

print("\nAfter balancing:")
print(balanced_df['label'].value_counts())
print("Total samples:", len(balanced_df))

# Save balanced dataset
balanced_df[['text', 'stars', 'label']].to_csv("../data/reviews_clean.csv", index=False)
print("\nSaved!")

Before balancing:
label
0    88751
1    11249
Name: count, dtype: int64

After balancing:
label
0    11249
1    11249
Name: count, dtype: int64
Total samples: 22498

Saved!
